# Adding a Custom Model to CLARYON

Step-by-step guide to implementing and registering a new model.

In [ ]:
# Step 1: Subclass ModelBuilder and register with @register

from __future__ import annotations
from pathlib import Path
from typing import Any

import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from joblib import dump, load as joblib_load

from claryon.models.base import ModelBuilder, InputType
from claryon.io.base import TaskType
from claryon.registry import register


@register("model", "custom_knn")
class CustomKNNModel(ModelBuilder):
    """K-Nearest Neighbors — custom model example."""

    def __init__(self, n_neighbors: int = 5, seed: int = 42, **kwargs: Any) -> None:
        self._n_neighbors = n_neighbors
        self._seed = seed
        self._model = None

    @property
    def name(self) -> str:
        return "custom_knn"

    @property
    def input_type(self) -> InputType:
        return InputType.TABULAR

    @property
    def supports_tasks(self) -> tuple[TaskType, ...]:
        return (TaskType.BINARY, TaskType.MULTICLASS)

    def fit(self, X: np.ndarray, y: np.ndarray, task_type: TaskType, **kwargs: Any) -> None:
        self._model = KNeighborsClassifier(n_neighbors=self._n_neighbors)
        self._model.fit(X, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self._model.predict(X)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self._model.predict_proba(X)

    def save(self, model_dir: Path) -> None:
        model_dir.mkdir(parents=True, exist_ok=True)
        dump(self._model, model_dir / "model.joblib")

    def load(self, model_dir: Path) -> None:
        self._model = joblib_load(model_dir / "model.joblib")

print("CustomKNNModel registered!")

In [ ]:
# Step 2: Verify registration
from claryon.registry import list_registered, get

models = list_registered("model")
print("Registered models:")
for name in sorted(models.keys()):
    print(f"  - {name}")

# Retrieve and instantiate
knn_cls = get("model", "custom_knn")
knn = knn_cls(n_neighbors=3)
print(f"\nInstantiated: {knn.name}, input_type={knn.input_type}")

In [ ]:
# Step 3: Use it directly
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, (iris.target > 0).astype(int), test_size=0.3, random_state=42
)

knn.fit(X_train, y_train, TaskType.BINARY)
preds = knn.predict(X_test)
print(f"BACC: {balanced_accuracy_score(y_test, preds):.3f}")

In [ ]:
# Step 4: Use it in a YAML config (reference by registry name)
config_example = """
models:
  - name: custom_knn       # Must match @register("model", "custom_knn")
    type: tabular
    params:
      n_neighbors: 7       # Passed to __init__
"""
print("YAML config for custom model:")
print(config_example)
print("Note: The pipeline discovers models via import. Place your file in")
print("claryon/models/classical/ or claryon/models/quantum/ and add the")
print("import to pipeline.py's _import_model_modules().")